# YOLOv5 Single-Image Test

This notebook loads a trained YOLOv5 model, runs inference on one image, saves the annotated result, and displays it.

In [1]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch

# Notebook should be opened from the YOLOv5 baseline directory.
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from models.common import DetectMultiBackend
from utils.general import check_img_size, non_max_suppression, scale_boxes
from utils.plots import Annotator
from utils.torch_utils import select_device


In [2]:
# Modify these paths as needed.
weights = "results/exp/weights/best.pt"
source = "dataset/bdd100k_selected/images/test/d4f13ad7-822cf350.jpg"

imgsz = 640
conf_thres = 0.25
iou_thres = 0.45
device_name = "cpu"
output_path = "results/testsample0001_with_anchorbox.jpg"


In [9]:
def run_inference(
    weights,
    source,
    imgsz=640,
    conf_thres=0.25,
    iou_thres=0.45,
    device_name="",
    output_path="result.jpg",
):
    device = select_device(device_name)
    model = DetectMultiBackend(weights, device=device)
    stride, names = model.stride, model.names

    imgsz = check_img_size(imgsz, s=stride)

    img = cv2.imread(str(source))
    if img is None:
        raise FileNotFoundError(f"Unable to read image: {source}")

    # Keep the original image for drawing.
    im = cv2.resize(img, (imgsz, imgsz))
    im = im.transpose(2, 0, 1)[::-1]
    im = torch.from_numpy(im.copy()).to(device)
    im = im.half() if model.fp16 else im.float()
    im /= 255.0
    im = im.unsqueeze(0)

    with torch.no_grad():
        pred = model(im, augment=False, visualize=False)

    pred = non_max_suppression(
        pred,
        conf_thres,
        iou_thres,
        max_det=1000,
    )

    example = next(iter(names.values())) if isinstance(names, dict) and names else "object"
    annotator = Annotator(
        img.copy(),
        line_width=2,
        font_size=15,
        pil=False,
        example=example,
    )

    detections = []

    for det in pred:
        if len(det):
            det[:, :4] = scale_boxes(
                im.shape[2:],
                det[:, :4],
                img.shape,
            ).round()

            for *xyxy, conf, cls in reversed(det):
                cls_id = int(cls)
                class_name = names[cls_id]
                label = f"{class_name} {conf:.2f}"
                annotator.box_label(xyxy, label, color=(255, 0, 0))

                detections.append({
                    "class_id": cls_id,
                    "class_name": class_name,
                    "confidence": float(conf),
                    "xyxy": [float(v) for v in xyxy],
                })

    result = annotator.im

    # Annotator may return a PIL image when pil=True.
    if not isinstance(result, type(img)):
        result = cv2.cvtColor(
            __import__("numpy").array(result),
            cv2.COLOR_RGB2BGR,
        )

    cv2.imwrite(str(output_path), result)
    print(f"Detected objects: {len(detections)}")
    print(f"Result saved to: {output_path}")

    return result, detections


In [10]:
result, detections = run_inference(
    weights=weights,
    source=source,
    imgsz=imgsz,
    conf_thres=conf_thres,
    iou_thres=iou_thres,
    device_name=device_name,
    output_path=output_path,
)


YOLOv5 🚀 2026-7-14 Python-3.11.15 torch-2.5.0+cu124 CPU

/root/autodl-tmp/26SummerHackathon_GenAI/baseline/models/experimental.py:98: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feat

Detected objects: 16
Result saved to: results/testsample0001_with_anchorbox.jpg


In [5]:
# Display the result inside the notebook.
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16, 9))
plt.imshow(result_rgb)
plt.axis("off")
plt.show()

In [6]:
# Inspect prediction details.
detections[:10]


[{'class_id': 4,
  'class_name': 'bus',
  'confidence': 0.27244794368743896,
  'xyxy': [847.0, 357.0, 890.0, 509.0]},
 {'class_id': 4,
  'class_name': 'bus',
  'confidence': 0.2729998826980591,
  'xyxy': [644.0, 419.0, 668.0, 471.0]},
 {'class_id': 4,
  'class_name': 'bus',
  'confidence': 0.2744719982147217,
  'xyxy': [696.0, 421.0, 725.0, 472.0]},
 {'class_id': 9,
  'class_name': 'traffic sign',
  'confidence': 0.28119590878486633,
  'xyxy': [200.0, 333.0, 372.0, 667.0]},
 {'class_id': 1,
  'class_name': 'rider',
  'confidence': 0.2965950071811676,
  'xyxy': [908.0, 470.0, 1056.0, 646.0]},
 {'class_id': 7,
  'class_name': 'bicycle',
  'confidence': 0.34536898136138916,
  'xyxy': [751.0, 198.0, 770.0, 235.0]},
 {'class_id': 2,
  'class_name': 'car',
  'confidence': 0.37087133526802063,
  'xyxy': [288.0, 363.0, 671.0, 720.0]},
 {'class_id': 4,
  'class_name': 'bus',
  'confidence': 0.4081120193004608,
  'xyxy': [741.0, 424.0, 778.0, 527.0]},
 {'class_id': 4,
  'class_name': 'bus',
  'c